# 09 — RAG corpus: inventory, download and text extraction

**Purpose.** Build the list of official documents the RAG may use, download them once, extract clean text, and record everything with hashes and licences. Contract and open questions: see notebook 08.

- **Inputs:** `configs/rag.yaml`, live pages at `stat.fi`, `tyomarkkinatori.fi` and `doria.fi`.
- **Outputs:**
  - `data/raw/rag/<doc_id>.<pdf|html>`: original files (git-ignored, some are *In Copyright*)
  - `data/raw/rag/text/<doc_id>.txt`: extracted text (git-ignored)
  - `data/manifests/rag_corpus_<timestamp>.json`: inventory, hashes, licences (committed)

**Safe by default.** `DRY_RUN = True` lists what would be downloaded and its size, and downloads nothing. Set it to `False` to fetch.

**Sources in version 1** (from notebook 08): A = Statistics Finland job vacancy releases, B = KEHA Employment Bulletins. Anything older than what those pages list is **not** collected here; see "Known gaps" at the end.

In [ ]:
# Mount Drive on Colab; skipped automatically when running locally.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    os.chdir("/content/drive/MyDrive/JobAI")
    os.environ["JOBAI_REPO"] = "/content/drive/MyDrive/JobAI"
except ImportError:
    pass

In [ ]:
# Only needed on a fresh runtime; the project requirements already list these.
# !pip install -q requests certifi trafilatura beautifulsoup4 lxml pyyaml pandas pypdf

In [ ]:
import os, re, json, time, hashlib, datetime as dt, importlib.metadata
from pathlib import Path
from urllib.parse import urljoin, urlparse, parse_qs, unquote, quote
import requests, certifi, yaml
import pandas as pd
from bs4 import BeautifulSoup

def _find_repo():
    env = os.environ.get("JOBAI_REPO")
    if env:
        return Path(env).resolve()
    p = Path.cwd().resolve()
    for cand in (p, *p.parents):
        if (cand / "configs" / "rag.yaml").is_file():
            return cand
    return p

REPO = _find_repo()
RAW_RAG = REPO / "data" / "raw" / "rag"
TEXT_DIR = RAW_RAG / "text"
MAN = REPO / "data" / "manifests"
for d in (RAW_RAG, TEXT_DIR, MAN):
    d.mkdir(parents=True, exist_ok=True)

CFG = yaml.safe_load(open(REPO / "configs" / "rag.yaml"))
NOW_UTC = dt.datetime.now(dt.timezone.utc)
TS = NOW_UTC.strftime("%Y%m%dT%H%M%SZ")

DRY_RUN = True          # False = actually download files
REQUEST_DELAY_S = 1.0   # be polite to public servers
TIMEOUT = 60
HEADERS = {"User-Agent": "JobAI-course-project/0.1 (research use; see repository README)"}
SESSION = requests.Session()
SESSION.headers.update(HEADERS)

def _fix_redirect(response, *args, **kwargs):
    # doria.fi redirects Finnish filenames (e.g. ...hein%c3%a426.pdf) with a raw
    # latin-1 byte in the Location header, which makes `requests` crash while it
    # follows the redirect. Rewrite the header as a valid, percent-encoded URL.
    loc = response.headers.get("Location")
    if loc:
        raw = loc.encode("latin-1", errors="ignore")
        try:
            text = raw.decode("utf-8")
        except UnicodeDecodeError:
            text = raw.decode("latin-1")
        response.headers["Location"] = quote(text, safe=":/?&=%#~@+,;!$'()*-._[]")
    return response

SESSION.hooks["response"].append(_fix_redirect)

def get(url, **kw):
    time.sleep(REQUEST_DELAY_S)
    r = SESSION.get(url, timeout=TIMEOUT, verify=certifi.where(), **kw)
    r.raise_for_status()
    return r

print("repo   :", REPO)
print("dry run:", DRY_RUN)

## Source A — Statistics Finland job vacancy releases

The release list on `stat.fi/en/statistics/atp` is rendered in the page for the **3 most recent** releases only; the full archive is loaded by JavaScript and cannot be read this way. Each release is a short HTML page.

**Licence:** CC BY 4.0 for texts, tables and graphs (Statistics Finland terms of use, checked 2026-09-21). Attribution is required.

In [ ]:
ATP_URL = "https://stat.fi/en/statistics/atp"

def clean_title(text):
    # Titles from these pages contain zero-width and non-breaking spaces.
    return re.sub(r"[\u200b\ufeff\xa0\s]+", " ", text or "").strip()
STAT_LICENCE = "CC BY 4.0 (Statistics Finland terms of use)"

def discover_statfin():
    soup = BeautifulSoup(get(ATP_URL).text, "lxml")
    rows, seen = [], set()
    for a in soup.find_all("a", href=True):
        if "/publication/" not in a["href"]:
            continue
        url = urljoin(ATP_URL, a["href"])
        if url in seen:
            continue
        seen.add(url)
        # The listing shows the release date (e.g. 20/08/2026) in a parent element
        # of the link; walk up until the nearest ancestor that contains one.
        published, node = None, a
        for _ in range(4):
            node = node.find_parent()
            if node is None:
                break
            m = re.search(r"(\d{2})/(\d{2})/(\d{4})", node.get_text(" ", strip=True))
            if m:
                published = f"{m.group(3)}-{m.group(2)}-{m.group(1)}"
                break
        rows.append({
            "doc_id": "statfin_atp_" + url.rstrip("/").split("/")[-1],
            "source_id": "A", "source_type": "statfin_release_page",
            "title": clean_title(a.get_text(" ", strip=True)), "language": "en",
            "published": published, "landing_url": url, "file_url": url,
            "format": "html", "licence": STAT_LICENCE,
        })
    return rows

statfin_rows = discover_statfin()
pd.DataFrame(statfin_rows)[["doc_id", "published", "title"]]

## Source B — KEHA Employment Bulletins

Työmarkkinatori lists bulletins from February 2025 with permanent `urn.fi` links (some are wrapped in an email-safety redirect and are unwrapped below). Each URN resolves to a record on DORIA, the national repository, which gives the PDF link, publication date, language and rights.

**Licence:** DORIA records these as **In Copyright 1.0**. That is not an open licence. The files are used here for a course project and are **not committed to git**. Confirm the terms with KEHA before any public release.

The English and Finnish pages are both read, so Finnish bulletins stay in Finnish (`configs/rag.yaml`).

In [ ]:
BULLETIN_PAGES = {
    "en": "https://tyomarkkinatori.fi/en/employment-and-statistics/evaluation-and-research/employment-bulletin",
    "fi": "https://tyomarkkinatori.fi/tyollisyys-ja-tilastot/arviointi-ja-tutkimus/tyollisyyskatsaus",
}
URN_RE = re.compile(r"URN:NBN:fi-fe\d+", re.I)

def unwrap(href):
    # Email-safety wrappers put the real link in the ?url= parameter.
    if "safelinks.protection.outlook.com" in href:
        href = unquote(parse_qs(urlparse(href).query).get("url", [href])[0])
    return href

def discover_bulletin_urns():
    found = {}
    for lang, page in BULLETIN_PAGES.items():
        soup = BeautifulSoup(get(page).text, "lxml")
        for a in soup.find_all("a", href=True):
            m = URN_RE.search(unquote(unwrap(a["href"])))
            if m:
                urn = m.group(0).upper().replace("URN:NBN:FI-FE", "URN:NBN:fi-fe")
                found.setdefault(urn, {"page_language": lang, "link_text": a.get_text(" ", strip=True)})
    return found

KEHA_LICENCE = "In Copyright (not open): (c) KEHA Centre"

def normalise_rights(raw):
    # DORIA rights are inconsistent: "In Copyright 1.0" on most records, a bare
    # "KEHA Centre" / "KEHA-keskus" or nothing on others. Treat all as not open.
    return KEHA_LICENCE

def read_doria(urn, page_language=None):
    landing = get(f"https://urn.fi/{urn}").url
    soup = BeautifulSoup(get(landing).text, "lxml")
    def meta(name):
        tag = soup.find("meta", attrs={"name": name})
        return tag["content"].strip() if tag and tag.get("content") else None
    pdf = meta("citation_pdf_url")
    return {
        "doc_id": "keha_bulletin_" + urn.split("fi-fe")[-1],
        "source_id": "B", "source_type": "keha_bulletin",
        "title": clean_title(meta("citation_title") or meta("DC.title")),
        "language": meta("citation_language") or meta("DC.language") or page_language,
        "published": meta("citation_date"),
        "landing_url": landing, "file_url": pdf, "format": "pdf",
        "licence": normalise_rights(meta("DC.rights")),
        "licence_raw": meta("DC.rights"),
        "urn": urn,
    }

bulletin_urns = discover_bulletin_urns()
print(len(bulletin_urns), "bulletin URNs found")
bulletin_rows, bulletin_errors = [], []
for urn, info in bulletin_urns.items():
    try:
        bulletin_rows.append(read_doria(urn, info["page_language"]))
    except Exception as exc:
        bulletin_errors.append({"urn": urn, "error": str(exc)[:200]})
print(len(bulletin_rows), "resolved;", len(bulletin_errors), "failed")
bulletin_rows[:2]

## Inventory

One table for both sources. Check it before downloading: dates, languages, licences, and rows with a missing PDF link.

In [ ]:
inventory = pd.DataFrame(statfin_rows + bulletin_rows)
inventory["published"] = inventory["published"].str[:10]
inventory = inventory.sort_values(["source_id", "published"], ascending=[True, False]).reset_index(drop=True)

problems = inventory[inventory["file_url"].isna() | inventory["published"].isna()]
print("documents:", len(inventory), "| with missing file_url or date:", len(problems))
display(inventory.groupby(["source_id", "language", "licence"]).agg(
    n=("doc_id", "size"), first=("published", "min"), last=("published", "max")).reset_index())
display(inventory[["doc_id", "source_id", "language", "published", "title"]])

### Date sanity check

The `published` date decides which documents retrieval may use (contract, rule 3). Some DORIA records carry a wrong year, for example a *December 2025* bulletin dated January 2025. A bulletin cannot be published before the month it describes, so those dates are flagged and `published_effective` is corrected by one year when that fits, otherwise set to the **first day after the bulletin month**. Both choices can only hide a document slightly too long, never expose a future one to an earlier forecast. Retrieval must use `published_effective`.

In [ ]:
MONTHS = {
    "january": 1, "february": 2, "march": 3, "april": 4, "may": 5, "june": 6, "july": 7,
    "august": 8, "september": 9, "october": 10, "november": 11, "december": 12,
    "tammikuu": 1, "helmikuu": 2, "maaliskuu": 3, "huhtikuu": 4, "toukokuu": 5, "kesäkuu": 6,
    "heinäkuu": 7, "elokuu": 8, "syyskuu": 9, "lokakuu": 10, "marraskuu": 11, "joulukuu": 12,
}

def bulletin_month_end(title):
    """First day after the month a bulletin describes, from its title; None if not parseable."""
    m = re.search(r"([A-Za-zäöå]+)\s+(20\d\d)", title or "")
    if not m or m.group(1).lower() not in MONTHS:
        return None
    month, year = MONTHS[m.group(1).lower()], int(m.group(2))
    return dt.date(year + (month == 12), month % 12 + 1, 1)

def effective_date(row):
    published = dt.date.fromisoformat(row["published"])
    if row["source_id"] != "B":
        return published, False
    earliest = bulletin_month_end(row["title"])
    if earliest and published < earliest:
        # Most bad records are one year early: try that correction first,
        # otherwise fall back to the first day after the bulletin month.
        try:
            fixed = published.replace(year=published.year + 1)
        except ValueError:
            fixed = None
        return (fixed if fixed and fixed >= earliest else earliest), True
    return published, False

pairs = inventory.apply(effective_date, axis=1)
inventory["published_effective"] = [p[0].isoformat() for p in pairs]
inventory["date_suspect"] = [p[1] for p in pairs]
print("suspect dates:", int(inventory.date_suspect.sum()))
display(inventory.loc[inventory.date_suspect, ["doc_id", "title", "published", "published_effective"]])

## Download

Sizes are checked with a HEAD request first. With `DRY_RUN = True` this cell only reports what would be fetched.

In [ ]:
def remote_size(url):
    try:
        time.sleep(REQUEST_DELAY_S)
        r = SESSION.head(url, timeout=TIMEOUT, allow_redirects=True, verify=certifi.where())
        return int(r.headers["Content-Length"]) if "Content-Length" in r.headers else None
    except Exception:
        return None

def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()

todo = inventory.dropna(subset=["file_url"]).copy()
todo["path"] = todo.apply(lambda r: RAW_RAG / f"{r.doc_id}.{r.format}", axis=1)
todo["cached"] = todo["path"].map(lambda p: p.is_file())

if DRY_RUN:
    todo["bytes"] = [remote_size(u) if not c else p.stat().st_size
                     for u, c, p in zip(todo.file_url, todo.cached, todo.path)]
    print("DRY RUN: nothing downloaded")
    print("files to fetch:", int((~todo.cached).sum()), "| already cached:", int(todo.cached.sum()))
    print("approx total MB:", round(todo["bytes"].fillna(0).sum() / 1e6, 1),
          "| unknown sizes:", int(todo["bytes"].isna().sum()))
    display(todo[["doc_id", "format", "bytes", "cached"]])
else:
    records = {}
    for row in todo.itertuples():
        if not row.cached:
            body = get(row.file_url).content
            row.path.write_bytes(body)
        else:
            body = row.path.read_bytes()
        records[row.doc_id] = {"sha256": sha256_bytes(body), "bytes": len(body),
                               "path": str(row.path.relative_to(REPO))}
    print("downloaded/verified", len(records), "files")

## Text extraction

HTML pages use `trafilatura` (keeps tables); PDFs use `pypdf`. Extracted text is saved one file per document. Any document with very little text is flagged, since scanned PDFs need OCR and this notebook does not do that.

In [ ]:
MIN_CHARS = 500

def extract_text(path, fmt):
    if fmt == "html":
        import trafilatura
        return trafilatura.extract(path.read_text(encoding="utf-8", errors="ignore"),
                                   include_tables=True, include_comments=False) or ""
    from pypdf import PdfReader
    reader = PdfReader(str(path))
    return "\n\n".join((page.extract_text() or "") for page in reader.pages)

extraction = []
if DRY_RUN:
    print("DRY RUN: skipping extraction (no files downloaded)")
else:
    for row in todo.itertuples():
        text = extract_text(row.path, row.format)
        (TEXT_DIR / f"{row.doc_id}.txt").write_text(text, encoding="utf-8")
        extraction.append({"doc_id": row.doc_id, "chars": len(text), "low_text": len(text) < MIN_CHARS})
    report = pd.DataFrame(extraction)
    print("documents:", len(report), "| low text (<%d chars):" % MIN_CHARS, int(report.low_text.sum()))
    display(report.sort_values("chars").head(10))

## Provenance manifest

Follows the manifest convention of notebooks 01 and 05. Written only after a real download.

In [ ]:
if DRY_RUN:
    print("DRY RUN: manifest not written")
else:
    manifest = {
        "name": "JobAI RAG corpus v1",
        "created_utc": NOW_UTC.isoformat(),
        "sources": {"A": "Statistics Finland job vacancy releases", "B": "KEHA Employment Bulletins"},
        "licence_notes": {
            "A": STAT_LICENCE,
            "B": "In Copyright (DORIA rights field, raw value kept per document as licence_raw); raw files are not committed.",
        },
        "documents": [
            {**{k: (None if pd.isna(v) else v) for k, v in rec.items()},
             **records.get(rec["doc_id"], {}),
             "chars": next((e["chars"] for e in extraction if e["doc_id"] == rec["doc_id"]), None)}
            for rec in inventory.to_dict(orient="records")
        ],
        "discovery_errors": bulletin_errors,
        "packages": {n: importlib.metadata.version(n) for n in
                     ["requests", "trafilatura", "beautifulsoup4", "pypdf", "pandas"]},
    }
    out = MAN / f"rag_corpus_{TS}.json"
    out.write_text(json.dumps(manifest, indent=2, ensure_ascii=False, default=str))
    print("wrote:", out)

## Known gaps (as of 2026-09-21)

- **Source A** covers only the 3 newest releases. Older releases are listed by JavaScript on `stat.fi`; releases before April 2022 are in the Finna archive. Extending this needs a seed list of URLs or a headless browser.
- **Source B** covers roughly February 2025 onwards. Earlier bulletins (reported 2013–2024) are on `tem.fi` and `tyollisyyskatsaus.fi` in other layouts and are not collected yet.
- **Licence for B is "In Copyright"**: raw files stay out of git. Confirm with KEHA before publishing the corpus or the index.
- **Sources C and D** (TEM forecast, Labour Force Barometer) are not in the `rag.yaml` whitelist and are not collected.
- **Scanned PDFs** are flagged as low-text but not OCR'd. None were found in the first run.
- **PDF text is noisy** (checked on the July 2026 bulletin). Map and chart labels come out as short fragments ("Uusimaa / 13,5"), words contain invisible soft hyphens (`Kanta\u00adHäme`), and numbers use decimal commas. Notebook 10 must strip soft hyphens and drop or merge label fragments before chunking. The useful narrative on vacancies is a short section of each bulletin, not the whole document.

**Next:** notebook 10 chunks and embeds this text and builds the index.